Set up / required imports

In [53]:
import sys
import os
import numpy as np
import pandas as pd
import lightgbm as lgb
from lightgbm import LGBMRegressor
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import warnings
from sklearn.metrics.pairwise import haversine_distances

warnings.filterwarnings("ignore")

In [3]:
# get the data
df = pd.read_csv("../data/unified_venue_pool.csv")

In [ ]:
df.drop(columns=["faiss_text"], inplace=True) # not needed here

Define LightGBM Feature Columns

These are the features LightGBM uses to score each (user, venue) pair.


They are split into three groups:

Venue static features —-> properties of the venue that do not change per user.

User-venue interaction features —-> computed at query time by combining user profile data with venue data. (changes per user and per query)

Context features —-> live signals (weather, time, day) injected at query time.

In [6]:
df.head(5)

,venue_id,name,data_source,venue_type,latitude,longitude,budget_encoded,budget_tier,meal_cost_for_one,cost_was_imputed,...,is_open_at_dinner_sun,is_open_at_breakfast_mon,is_open_at_breakfast_tue,is_open_at_breakfast_wed,is_open_at_breakfast_thu,is_open_at_breakfast_fri,is_open_at_breakfast_sat,is_open_at_breakfast_sun,weekend_open_hours,weekday_open_hours
0,d3068ce4-da78-41f6-9184-c6bf5dacac85,Maze Specialty Coffee,ae,dining,25.09056,55.22854,0,low,47.5,0,...,1,1,1,1,1,1,1,1,17,17
1,56f201bf-d7b7-4f25-bc70-06c63fcb7506,Loca,ae,dining,25.23349,55.26342,2,high,200.0,0,...,1,0,0,0,0,0,0,0,14,14
2,34e582b3-2162-4468-af17-bbe4039f1667,Baofriend,ae,dining,25.11904,55.37809,0,low,50.0,0,...,1,0,0,0,0,0,0,0,11,13
3,b84bcdfb-bc23-4aa2-8ef3-916816a8bca3,MayaBay,ae,dining,25.13666,55.18666,2,high,237.5,0,...,1,0,0,0,0,0,0,0,15,15
4,4ccfe9bf-d54d-43bc-85e2-2589ba44d867,Doner Deli,ae,dining,25.23392,55.35696,1,medium,72.5,1,...,1,0,0,0,0,0,0,0,13,13


venue static features / no computation needed just select the right columns

In [7]:
# extract all category columns
CUISINE_CATS = [
    col.replace("cat_", "")
    for col in df.columns
    if col.startswith("cat_")
]

In [8]:
print(CUISINE_CATS)

['american', 'arabic', 'asian', 'bar', 'cafe', 'chinese', 'european', 'fastfood', 'french', 'grills', 'healthy', 'indian', 'international', 'italian', 'japanese', 'lebanese', 'mediterranean', 'mexican', 'seafood', 'turkish', 'beach_waterfront', 'cultural_centre', 'gallery', 'heritage_site', 'library', 'museum', 'park_attraction']


In [ ]:
VENUE_STATIC_FEATURES = (
    # Budget and cost
    ['meal_cost_for_one', 'budget_encoded'] +
    # Amenities used as hard signals: alcohol, shisha, outdoor seating
    ['serves_alcohol', 'has_shisha', 'has_outdoor_seating'] +
    # Venue type signals
    ['is_chain'] +
    # Operating hours
    ['open_duration_fri', 'is_open_at_dinner_fri', 'is_open_at_breakfast_fri',
     'days_open_at_dinner', 'weekend_open_hours', 'weekday_open_hours',
     'weekend_open_boost'] +
    # Category binary columns
    CUISINE_CATS
)

User-venue interaction features (computed per query)

In [10]:
# These do not exist in the venue pool — they are derived at query time
# For training, i will simulate them using survey respondent profiles.
USER_VENUE_FEATURES = [
    'haversine_distance_km',  # computed from user location + venue lat/lon
    'budget_delta',           # user budget_per_outing - venue meal_cost_for_one
    'budget_match',           # 1 if venue budget_encoded == user budget_level
    'category_overlap_score', # how many user preferred cats match venue cats
    'avoids_alcohol_conflict', # 1 if user avoids alcohol AND venue serves_alcohol=1
    'avoids_shisha_conflict',  # 1 if user avoids shisha AND venue has_shisha=1
]

Context features (live signals, simulated here)


In [11]:
CONTEXT_FEATURES = [
    'outdoor_ok',             
    'is_weekend',            
    'is_dinner_request',      # 1 if user's requested time slot is evening
    'is_holiday',             # 1 if today is a UAE public holiday
]

Total LightGBM features: 50
  Venue static:     40
  User-venue:       6
  Context:          4
  Cuisine categories: 27


Haversine Distance Function

Computed at query time from the user's current location and each venue's stored coordinates

Never stored as a static column —-> it changes per user, per query

In [17]:
def haversine_km(lat1, lon1, lat2, lon2):
    """
    Compute distance between two geographic points in kilometers.
    """

    point1 = np.radians([lat1, lon1])
    point2 = np.radians([lat2, lon2])

    distance_rad = haversine_distances([point1], [point2])[0][0]

    earth_radius_km = 6371 # to convert to km

    return distance_rad * earth_radius_km

In [20]:
# Test: distance from Marina Plaza to Dubai Mall
dist = haversine_km(25.0805, 55.1435, 25.1972, 55.2797)
print(f"Test distance from Marina Plaza to Dubai Mall: {dist:.2f} km  (expected ~20 km)")

Test distance from Marina Plaza to Dubai Mall: 18.88 km  (expected ~20 km)


Utilizing user data from the survery responses

In [28]:
SURVEY_PATH = "/Users/mahra/ai-agent-project/AI_lifestyle_agent/ml/data/augmented_combined.csv"

In [29]:
# load survey respondents
SURVEY_AVAILABLE = os.path.exists(SURVEY_PATH)
print(f"Survey file available: {SURVEY_AVAILABLE}")

survey = pd.read_csv(SURVEY_PATH)
print(f"Survey loaded: {survey.shape}")
print(f"User profiles: {len(survey)}")

Survey file available: True
Survey loaded: (1101, 75)
User profiles: 1101


Compute a synthetic relevance score for a (user, venue) pair.
Score range: 0.0 (irrelevant) to 1.0 (perfect match)

Components:
    category_overlap (40% weight) — do venue categories match user preferences?
    budget_match     (25% weight) — does venue price fit user budget?
    distance_score   (20% weight) — is the venue close enough?
    constraint_ok    (15% weight) — does venue violate any hard constraints?

Why these weights:
    Survey respondents ranked cost, distance, and quality as top-3 factors.
    Category is the most direct preference signal so it gets highest weight.
    Constraint violations (alcohol when user avoids it) are penalised heavily.

In [30]:
restaurant_path = '/Users/mahra/ai-agent-project/AI_lifestyle_agent/ml/data/unified_venue_pool.csv'

In [43]:
venues = pd.read_csv(restaurant_path)

In [42]:
#  Distance thresholds from travel_distance_encoded 
# 0 = nearby only  --> 3 km
# 1 = short        --> 7 km
# 2 = medium       --> 15 km
# 3 = long         --> 25 km
# 4 = anywhere     --> no penalty
TRAVEL_DISTANCE_THRESHOLDS = {0: 3, 1: 7, 2: 15, 3: 25, 4: 999}

In [44]:
#  Category columns in the venue pool 
VENUE_CAT_COLS = [c.replace('cat_', '') for c in venues.columns
                  if c.startswith('cat_')]

In [45]:
#  Exclusion --> venue category mapping 
# Maps each excl_* user flag to the venue cat_* columns it blocks.
# When excl_nightlife=1, venues tagged cat_bar=1 should score 0.
EXCLUSION_TO_VENUE_CATS = {
    'excl_nightlife':     ['cat_bar'],
    'excl_cultural':      ['cat_museum','cat_gallery','cat_heritage_site',
                           'cat_cultural_centre','cat_library'],
    'excl_water':         ['cat_beach_waterfront'],
    'excl_outdoor_travel':['cat_park_attraction','cat_beach_waterfront'],
}

In [50]:
USER_PROFILE_FEATURES = [
    'budget_encoded', 'travel_distance_encoded', 'weather_pref_encoded',
    'pref_morning', 'pref_midday', 'pref_afternoon', 'pref_evening',
    'pref_late_night', 'diet_halal', 'diet_vegetarian', 'diet_vegan',
    'diet_gluten_free', 'social_friends', 'social_family', 'social_partner',
    'social_alone', 'factor_cost', 'factor_distance', 'factor_quality',
    'factor_comfort', 'adventure_level_encoded', 'currently_saving_money',
    'going_out_frequency_encoded', 'activity_duration_encoded',
    'excl_nightlife', 'excl_cultural', 'excl_water', 'excl_outdoor_travel',
    'has_exceptions',
]

In [55]:
ALL_FEATURES = VENUE_STATIC_FEATURES + USER_VENUE_FEATURES + CONTEXT_FEATURES

print(f"Total LightGBM features: {len(ALL_FEATURES)}")
print(f"  Venue static:     {len(VENUE_STATIC_FEATURES)}")
print(f"  User-venue:       {len(USER_VENUE_FEATURES)}")
print(f"  Context:          {len(CONTEXT_FEATURES)}")
print(f"  Cuisine categories: {len(CUISINE_CATS)}")

Total LightGBM features: 50
  Venue static:     40
  User-venue:       6
  Context:          4
  Cuisine categories: 27


Compute a synthetic relevance score for a (user, venue) pair.
Score range: 0.0 (irrelevant) to 1.0 (perfect match)


Components and weights (informed by actual factor columns in user data):
    budget_match     (30%) — user budget_encoded vs venue budget_encoded
    distance_score   (25%) — haversine vs user travel_distance_encoded preference
    constraint_ok    (25%) — halal, exclusions, dietary violations
    time_slot_match  (10%) — user pref_* columns vs venue open hours
    weather_match    (10%) — user weather_pref_encoded vs has_outdoor_seating

Why these weights vs the old ones:
    - Category overlap dropped: users have no explicit cuisine preference column.
        The cat_* columns exist only on venues, not on users.
    - Budget promoted to 30%: budget_encoded is now a direct 0/1/2 match on
        both sides, and factor_cost is the #2 stated decision factor (43% of users).
    - Constraint promoted to 25%: diet_halal affects 38% of users and is a
        HARD constraint. Getting it wrong destroys trust.
    - Distance stays prominent: factor_distance is the #1 stated decision
        factor (72% of users flagged it), and travel_distance_encoded gives us
        a per-user threshold rather than a global 3km cutoff.

In [48]:
def compute_relevance_score(user_row: pd.Series, venue_row: pd.Series) -> float:

    # budget match (0 to 1) 
    # exact match = 1.0
    # one tier off = 0.5. 
    # two tiers off = 0.0.
    user_budget  = int(user_row.get('budget_encoded', 1))
    venue_budget = int(venue_row.get('budget_encoded', 1))
    budget_diff  = abs(user_budget - venue_budget)
    budget_score = {0: 1.0, 1: 0.5, 2: 0.0}[budget_diff]

    #  Distance score (0 to 1) 
    # Uses the user's stated travel_distance_encoded to set the threshold,
    # not a hardcoded global cutoff. 
    # A user who says "anywhere" (4) gets no distance penalty. 
    # A user who says "nearby only" (0) penalises anything beyond 3km heavily.
    user_lat  = float(user_row.get('user_latitude',  25.1))
    user_lon  = float(user_row.get('user_longitude', 55.2))
    venue_lat = float(venue_row.get('latitude', float('nan')))
    venue_lon = float(venue_row.get('longitude', float('nan')))
    travel_pref = int(user_row.get('travel_distance_encoded', 2))
    max_km = TRAVEL_DISTANCE_THRESHOLDS[travel_pref]

    dist_km = haversine_km(user_lat, user_lon, venue_lat, venue_lon)
    if pd.isna(dist_km):
        distance_score = 0.3       # unknown location: neutral
    elif travel_pref == 4:
        distance_score = 1.0       # user explicitly said "anywhere"
    elif dist_km <= max_km * 0.33:
        distance_score = 1.0       # well within their threshold
    elif dist_km <= max_km * 0.66:
        distance_score = 0.6       # within threshold
    elif dist_km <= max_km:
        distance_score = 0.3       # right at their limit
    else:
        distance_score = 0.0       # exceeds their stated travel willingness

    # constraint score (0 or 1 — hard violations give 0) 
    # these are HARD constraints
    # violating any one --> score = 0
    # the model must never recommend an alcohol-serving venue to a halal user
    constraint_score = 1.0

    # halal: 38% of users so alcohol-serving venues are incompatible
    if user_row.get('diet_halal', 0) == 1 and venue_row.get('serves_alcohol', 0) == 1:
        constraint_score = 0.0

    # vegetarian/vegan: cannot recommend grills/steakhouses as primary category
    if constraint_score > 0:
        if user_row.get('diet_vegetarian', 0) == 1 or user_row.get('diet_vegan', 0) == 1:
            if venue_row.get('cat_grills', 0) == 1 and venue_row.get('cat_healthy', 0) == 0:
                constraint_score = 0.0

    # exclusion flags: check if venue type matches an excluded category
    if constraint_score > 0:
        for excl_col, blocked_cat_cols in EXCLUSION_TO_VENUE_CATS.items():
            if user_row.get(excl_col, 0) == 1:
                if any(venue_row.get(cat, 0) == 1 for cat in blocked_cat_cols):
                    constraint_score = 0.0
                    break

    # Time slot match (0 to 1) --> does the venue's opening hours cover the user's preferred time of day?
    # check Friday as the representative day 
    time_score = 0.5   # neutral default when data is unavailable
    if constraint_score > 0:   # skip if already violated
        user_prefers_evening   = int(user_row.get('pref_evening', 0))
        user_prefers_morning   = int(user_row.get('pref_morning', 0))
        venue_open_dinner  = int(venue_row.get('is_open_at_dinner_fri', 1))
        venue_open_morning = int(venue_row.get('is_open_at_breakfast_fri', 1))

        if user_prefers_evening and venue_open_dinner:
            time_score = 1.0
        elif user_prefers_morning and venue_open_morning:
            time_score = 1.0
        elif user_prefers_evening and not venue_open_dinner:
            time_score = 0.0   # user wants evenings, venue closes early
        elif user_prefers_morning and not venue_open_morning:
            time_score = 0.0   # user wants mornings, venue opens late

    #  Weather match (0 to 1) 
    # weather_pref_encoded: 0=indoor only, 1=outdoor only, 2=mixed, 3=no preference
    # We match against has_outdoor_seating (only 2% of venues — sparse but real).
    weather_pref = int(user_row.get('weather_pref_encoded', 3))
    venue_outdoor = int(venue_row.get('has_outdoor_seating', 0))

    if weather_pref == 3:
        weather_score = 1.0    # no preference → all venues equally fine
    elif weather_pref == 2:
        weather_score = 0.8    # mixed preference → slight preference for outdoor
    elif weather_pref == 1:    # prefers outdoor
        weather_score = 1.0 if venue_outdoor else 0.4
    elif weather_pref == 0:    # prefers indoor
        weather_score = 0.4 if venue_outdoor else 1.0
    else:
        weather_score = 0.8

    #  Weighted composite
    score = (
        0.30 * budget_score +
        0.25 * distance_score +
        0.25 * constraint_score +
        0.10 * time_score +
        0.10 * weather_score
    )
    return round(min(score, 1.0), 4)

In [49]:
# quick test
sample_user  = survey.iloc[0]
sample_venue = venues.iloc[0]
test_score   = compute_relevance_score(sample_user, sample_venue)
print(f"Test relevance score (user 0, venue 0): {test_score:.4f}")

Test relevance score (user 0, venue 0): 0.7400


Start Training:

For each user, sample a random subset of venues (not all 360 — that would be
slow and produce too many near-zero scores that drown out signal).


Strategy: for each user, take:
  - 10 "positive" venues (highest relevance by our scoring function)
  - 10 "negative" venues (random sample of low-relevance venues)
  
This gives a balanced training set with real contrast.

 How to compute a synthetic relevance score between a user and a venue.

In a real recommender system, models like LightGBM usually learn from
user behaviour such as:
    - clicks
    - bookings
    - ratings
    - saves/favourites
    - dwell time

However, in cold-start systems (new systems with no interaction data),we do not yet have those labels.

This function creates SYNTHETIC labels using domain knowledge.

In other words:
    "Based on our understanding of user preferences,how relevant should this venue be?"

These scores can later be used to:
    1. train a LightGBM ranking model
    2. evaluate recommendation quality
    3. create a baseline recommender
    4. bootstrap the system before real user data exists

Final score range:
    0.0 = completely irrelevant
    1.0 = highly relevant

Recommendation Factors

1. Category overlap  (40%)
    Does the venue cuisine/category match what the user likes?

2. Budget match      (25%)
    Is the venue affordable for the user?

3. Distance score    (20%)
    Is the venue geographically close enough?

4. Constraint score  (15%)
    Does the venue violate hard user preferences?
    Example:
        - user avoids alcohol
        - user avoids shisha

Why these weights?

These weights reflect domain assumptions from survey findings:
- category preference is usually the strongest signal
- budget and distance strongly affect real-world decisions
- constraints are important for recommendation safety/trust

Important ML Note
This function should ideally be used as:
    - a TRAINING LABEL generator
    - NOT as a direct feature itself

The underlying components (distance, budget difference, etc.)
should be individual model features.

The final composite score acts as the target the model learns to predict.

In [ ]:
# build synthetic training pairs 
# Full cross-product: 1,101 users × 11,603 venues = 12.7M pairs → too slow.
# Strategy per user:
#   1. Score a random sample of 200 venues using compute_relevance_score
#   2. Keep the top 10 by score  (positives — good matches)
#   3. Keep the bottom 10 by score (negatives — poor matches)
#   4. Discard the middle — it adds noise without signal
# Result: 1,101 × 20 = ~22,020 training rows with strong label contrast.print("Building synthetic training pairs...")
print(f"Users: {len(survey)} | Venues: {len(venues)} | Sample per user: 200 → keep 20")

np.random.seed(42)
training_rows    = []
SAMPLE_PER_USER  = 200   # venues scored per user before selecting top/bottom
KEEP_TOP         = 10    # highest-scored venues kept as positives
KEEP_BOTTOM      = 10    # lowest-scored venues kept as negatives

for user_idx, user in survey.iterrows():

    sampled_venues = venues.sample(
        n=min(SAMPLE_PER_USER, len(venues)),
        random_state=user_idx
    )

    scored = []
    for venue_idx, venue in sampled_venues.iterrows():
        label = compute_relevance_score(user, venue)
        scored.append((venue_idx, label))

    scored.sort(key=lambda x: x[1], reverse=True)
    selected = scored[:KEEP_TOP] + scored[-KEEP_BOTTOM:]

    for venue_idx, label in selected:
        venue = venues.loc[venue_idx]
         # user-venue interaction features 
        # these are computed per pair (they change per user and cannot be stored statically in the venue pool)

        #  user-venue interaction features 
        dist_km = haversine_km(
            float(user['user_latitude']),
            float(user['user_longitude']),
            float(venue.get('latitude', float('nan'))),
            float(venue.get('longitude', float('nan'))),
        )

        user_budget  = int(user['budget_encoded'])
        venue_budget = int(venue.get('budget_encoded', 1))
        budget_diff  = abs(user_budget - venue_budget)
        budget_score = {0: 1.0, 1: 0.5, 2: 0.0}[budget_diff]

        travel_pref = int(user['travel_distance_encoded'])
        max_km      = TRAVEL_DISTANCE_THRESHOLDS[travel_pref]

        if pd.isna(dist_km):
            distance_score = 0.3
        elif travel_pref == 4:
            distance_score = 1.0
        elif dist_km <= max_km * 0.33:
            distance_score = 1.0
        elif dist_km <= max_km * 0.66:
            distance_score = 0.6
        elif dist_km <= max_km:
            distance_score = 0.3
        else:
            distance_score = 0.0
        
        # hard constraint flags (used both in the label and as features)
        diet_halal_conflict     = int(user['diet_halal'] == 1
                                      and venue.get('serves_alcohol', 0) == 1)
        diet_veg_conflict       = int((user['diet_vegetarian'] == 1
                                       or user['diet_vegan'] == 1)
                                      and venue.get('cat_grills', 0) == 1
                                      and venue.get('cat_healthy', 0) == 0)
        excl_nightlife_conflict = int(user['excl_nightlife'] == 1
                                      and venue.get('cat_bar', 0) == 1)
        excl_cultural_conflict  = int(user['excl_cultural'] == 1
                                      and any(venue.get(c, 0) == 1 for c in [
                                          'cat_museum', 'cat_gallery',
                                          'cat_heritage_site',
                                          'cat_cultural_centre', 'cat_library'
                                      ]))
        excl_water_conflict     = int(user['excl_water'] == 1
                                      and venue.get('cat_beach_waterfront', 0) == 1)

        constraint_score = 0.0 if any([
            diet_halal_conflict, diet_veg_conflict,
            excl_nightlife_conflict, excl_cultural_conflict,
            excl_water_conflict,
        ]) else 1.0

        user_pref_evening  = int(user['pref_evening'])
        user_pref_morning  = int(user['pref_morning'])
        venue_open_dinner  = int(venue.get('is_open_at_dinner_fri', 1))
        venue_open_morning = int(venue.get('is_open_at_breakfast_fri', 1))

        # time slot match
        if user_pref_evening and venue_open_dinner:
            time_score = 1.0
        elif user_pref_morning and venue_open_morning:
            time_score = 1.0
        elif user_pref_evening and not venue_open_dinner:
            time_score = 0.0
        elif user_pref_morning and not venue_open_morning:
            time_score = 0.0
        else:
            time_score = 0.5
            
        # weather match
        weather_pref  = int(user['weather_pref_encoded'])
        venue_outdoor = int(venue.get('has_outdoor_seating', 0))

        if weather_pref == 3:
            weather_score = 1.0
        elif weather_pref == 2:
            weather_score = 0.8
        elif weather_pref == 1:
            weather_score = 1.0 if venue_outdoor else 0.4
        else:
            weather_score = 0.4 if venue_outdoor else 1.0

        #  simulated context features 
        # in production these come from weather service + calendar at request time.
        # just making it up here
        outdoor_ok        = 1   # April conditions — outdoor allowed (assume)
        is_weekend        = 1   # simulating Friday evening request (assume)
        is_dinner_request = int(user['pref_evening'])
        is_holiday        = 0

        #  assemble training row 
        row = {
            'user_id':                  user_idx,
            'venue_id':                 venue['venue_id'],
            'label':                    label,
            'synthetic_label':          1,
            
            # interaction features
            'haversine_distance_km':    dist_km,
            'budget_diff':              budget_diff,
            'budget_score':             budget_score,
            'distance_score':           distance_score,
            'constraint_score':         constraint_score,
            'time_score':               time_score,
            'weather_score':            weather_score,
            'diet_halal_conflict':      diet_halal_conflict,
            'diet_veg_conflict':        diet_veg_conflict,
            'excl_nightlife_conflict':  excl_nightlife_conflict,
            'excl_cultural_conflict':   excl_cultural_conflict,
            'excl_water_conflict':      excl_water_conflict,
           
            # context features
            'outdoor_ok':               outdoor_ok,
            'is_weekend':               is_weekend,
            'is_dinner_request':        is_dinner_request,
            'is_holiday':               is_holiday,
        }

        # venue static features (direct from venue pool)
        for feat in VENUE_STATIC_FEATURES:
            row[feat] = venue.get(feat, 0)
            
        # user profile features (from user row)
        for feat in USER_PROFILE_FEATURES:
            row[feat] = user.get(feat, 0)

        training_rows.append(row)

    if (user_idx + 1) % 100 == 0:
        print(f"  {user_idx + 1}/{len(survey)} users processed...")


train_df = pd.DataFrame(training_rows)
print(f"\nTraining pairs built: {len(train_df):,}")
print(f"\nLabel distribution:")
print(f"  Mean:                     {train_df['label'].mean():.3f}")
print(f"  Std:                      {train_df['label'].std():.3f}")
print(f"  >= 0.7 (strong positive): {(train_df['label'] >= 0.7).sum()}")
print(f"  0.3–0.7 (neutral):        {((train_df['label'] >= 0.3) & (train_df['label'] < 0.7)).sum()}")
print(f"  < 0.3 (negative):         {(train_df['label'] < 0.3).sum()}")
print(f"\nConstraint violations: {(train_df['constraint_score'] == 0).sum()}")
print(f"Diet halal conflicts:   {train_df['diet_halal_conflict'].sum()}")
print(f"Excl nightlife:         {train_df['excl_nightlife_conflict'].sum()}")

Users: 1101 | Venues: 11603 | Sample per user: 200 → keep 20
  100/1101 users processed...
  200/1101 users processed...
  300/1101 users processed...
  400/1101 users processed...
  500/1101 users processed...
  600/1101 users processed...
  700/1101 users processed...
  800/1101 users processed...
  900/1101 users processed...
  1000/1101 users processed...
  1100/1101 users processed...

Training pairs built: 22,020

Label distribution:
  Mean:                     0.653
  Std:                      0.221
  >= 0.7 (strong positive): 10916
  0.3–0.7 (neutral):        10150
  < 0.3 (negative):         954

Constraint violations: 3802
Diet halal conflicts:   3132
Excl nightlife:         217


Train LightGBM Ranker

We use LightGBM's LGBMRegressor to predict relevance scores

Q1) Why regression and not classification:

We want a continuous score to rank 20 candidates. A binary classifier would
only tell us "relevant" or "not relevant" — we need a ranked order.
Regression on the 0–1 relevance score naturally produces a ranking signal.

Q2) Why LightGBM and not XGBoost or a neural net:

- Handles NaN natively (no imputation needed for venues with missing coordinates)
- Fast on CPU (no GPU needed for 360 venues)
- Natively supports categorical features (budget_encoded, location_cluster)
- Feature importance is interpretable — critical for explaining recommendations

**Hyperparameter choices:**
Starting with sensible defaults. 
After we collect real interaction data, run a proper hyperparameter search using the evaluation metrics from the path `ml/evaluation/evaluate.ipynb`


In [60]:
#  Diagnose before splitting 
missing = [f for f in ALL_FEATURES if f not in train_df.columns]
extra   = [c for c in train_df.columns
           if c not in ALL_FEATURES
           and c not in ('user_id', 'venue_id', 'label', 'synthetic_label')]

print(f"Features in ALL_FEATURES missing from train_df: {missing}")
print(f"Columns in train_df not in ALL_FEATURES:        {extra}")

Features in ALL_FEATURES missing from train_df: ['budget_delta', 'budget_match', 'category_overlap_score', 'avoids_alcohol_conflict', 'avoids_shisha_conflict']
Columns in train_df not in ALL_FEATURES:        ['budget_diff', 'budget_score', 'distance_score', 'constraint_score', 'time_score', 'weather_score', 'diet_halal_conflict', 'diet_veg_conflict', 'excl_nightlife_conflict', 'excl_cultural_conflict', 'excl_water_conflict', 'travel_distance_encoded', 'weather_pref_encoded', 'pref_morning', 'pref_midday', 'pref_afternoon', 'pref_evening', 'pref_late_night', 'diet_halal', 'diet_vegetarian', 'diet_vegan', 'diet_gluten_free', 'social_friends', 'social_family', 'social_partner', 'social_alone', 'factor_cost', 'factor_distance', 'factor_quality', 'factor_comfort', 'adventure_level_encoded', 'currently_saving_money', 'going_out_frequency_encoded', 'activity_duration_encoded', 'excl_nightlife', 'excl_cultural', 'excl_water', 'excl_outdoor_travel', 'has_exceptions']


In [59]:
#  train / test split 
# split by user_id so no user appears in both train and test 
unique_users = train_df['user_id'].unique()
n_test_users = max(1, int(len(unique_users) * 0.2))   # 20% of users for test

np.random.seed(42)
test_users  = np.random.choice(unique_users, n_test_users, replace=False)
train_users = [u for u in unique_users if u not in test_users]

train_set = train_df[train_df['user_id'].isin(train_users)]
test_set  = train_df[train_df['user_id'].isin(test_users)]

X_train = train_set[ALL_FEATURES].values
y_train = train_set['label'].values
X_test  = test_set[ALL_FEATURES].values
y_test  = test_set['label'].values

print(f"Train: {len(train_set)} pairs ({len(train_users)} users)")
print(f"Test:  {len(test_set)} pairs ({len(test_users)} users)")


KeyError: "['budget_delta', 'budget_match', 'category_overlap_score', 'avoids_alcohol_conflict', 'avoids_shisha_conflict'] not in index"

In [ ]:

# ── Declare categorical features ──────────────────────────────────────────────
# LightGBM treats these as unordered categories rather than continuous numbers.
# budget_encoded (0/1/2) should NOT be treated as 0 < 1 < 2 in a linear sense —
# it is an ordinal category, and tree splits on it are category-aware.
cat_feature_names = ['budget_encoded', 'location_cluster']
cat_feature_names = [f for f in cat_feature_names if f in ALL_FEATURES]
cat_feature_indices = [ALL_FEATURES.index(f) for f in cat_feature_names]

# ── Train ─────────────────────────────────────────────────────────────────────
model = LGBMRegressor(
    n_estimators=200,          # number of trees — increase if underfitting
    learning_rate=0.05,        # smaller = more conservative = more trees needed
    num_leaves=31,             # controls model complexity (max 2^depth-1)
    max_depth=6,               # prevents overfitting on small dataset
    min_child_samples=5,       # minimum samples per leaf — important for small N
    subsample=0.8,             # row sampling per tree (bagging)
    colsample_bytree=0.8,      # feature sampling per tree
    reg_alpha=0.1,             # L1 regularisation
    reg_lambda=0.1,            # L2 regularisation
    random_state=42,
    verbose=-1,
)

model.fit(
    X_train, y_train,
    categorical_feature=cat_feature_indices,
    eval_set=[(X_test, y_test)],
    callbacks=[lgb.early_stopping(20, verbose=False), lgb.log_evaluation(50)],
)

print(f"\nBest iteration: {model.best_iteration_}")
print(f"Train MSE: {((model.predict(X_train) - y_train)**2).mean():.4f}")
print(f"Test MSE:  {((model.predict(X_test)  - y_test )**2).mean():.4f}")
